# 12a — Download all cutouts for Gaia-stable diaObjects

## Purpose

Download and cache on disk the three cutout stamps (Science, Template, Difference)
**and** forced photometry for every diaSource belonging to the **Gaia-stable**
diaObjects identified in
`09b_psfFlux_scienceFlux_templateFlux_GaiaDR3matching.ipynb`.

The list of target diaObjectIds is read from the CSV produced by notebook 09b,
filtered to `gaia_status == "Gaia stable"`.
All downloads are delegated to `fink_download_full_cutouts.py`
(imported as a module from the same directory).

## Inputs

| File | Produced by |
|------|-------------|
| `figs_FINK_BLOCK_LC_09b/median_flux_stats_gaia_star_stable_hq.csv` | notebook 09b |
| `fink_download_full_cutouts.py` | this directory |

## Output (one directory per diaObject)

```
fullcutouts_{diaObjectId}/
    manifest.csv / manifest.parquet        ← diaSource metadata + dipole columns
    manifest_fp.csv / manifest_fp.parquet  ← forced photometry
    cutouts/
        {diaSourceId}_{band}_Science.npy
        {diaSourceId}_{band}_Template.npy
        {diaSourceId}_{band}_Difference.npy
```

---
- **Author:** Sylvie Dagoret-Campagne — IJCLab / IN2P3 / CNRS — Université Paris-Saclay
- **Creation date:** 2026-05-13
- **Subject:** Fink/LSST DIA — Dipole hypothesis — cutout + fp download


## 1. Imports & configuration

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
# ── Import download function from the script in the same directory ────────────
NB_DIR = os.path.abspath(".")
if NB_DIR not in sys.path:
    sys.path.insert(0, NB_DIR)

from fink_download_full_cutouts import download_full_cutouts

print(f"download_full_cutouts imported from {NB_DIR}/fink_download_full_cutouts.py")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────

# CSV produced by notebook 09b — contains only gaia_star_stable_hq objects
FILE_STATS = os.path.join(
    "figs_FINK_BLOCK_LC_09b",
    "median_flux_stats_gaia_star_stable_hq.csv",
)

# Set to True to re-download even if fullcutouts_{id}/ already exists
FORCE_REDOWNLOAD = False

print(f"Input CSV : {os.path.abspath(FILE_STATS)}")

## 2. Read the target diaObjectId list

In [ ]:
if not os.path.exists(FILE_STATS):
    raise FileNotFoundError(
        f"{FILE_STATS} not found.\n"
        "Run notebook 09b_psfFlux_scienceFlux_templateFlux_GaiaDR3matching.ipynb first."
    )

df_stable = pd.read_csv(FILE_STATS).sort_values("rank").reset_index(drop=True)

print(f"Gaia-stable objects to download : {len(df_stable)}")
print()
print(
    df_stable[["rank", "diaObjectId", "n_alerts", "gaia_origin", "gaia_G_mag", "gaia_status"]].to_string(
        index=False
    )
)

## 3. Download cutouts + forced photometry

For each object, `download_full_cutouts()` :
1. fetches all diaSources via `/api/v1/sources` → `manifest.csv`
2. downloads the 3 stamp arrays per diaSource → `cutouts/*.npy`
3. fetches forced photometry via `/api/v1/fp` → `manifest_fp.csv`

Already-downloaded objects are skipped (idempotent re-runs).

In [ ]:
results = []

for _, row in df_stable.iterrows():
    obj_id = int(row["diaObjectId"])
    n_alerts = int(row["n_alerts"])
    origin = row["gaia_origin"]
    g_mag = row["gaia_G_mag"]
    outdir = Path(f"fullcutouts_{obj_id}")

    manifest_path = outdir / "manifest.csv"
    already_done = manifest_path.exists() and not FORCE_REDOWNLOAD

    print(f"\n{'=' * 60}")
    print(f"diaObjectId = {obj_id}  |  Gaia G = {g_mag:.3f}  |  origin = {origin}  |  n_alerts = {n_alerts}")

    if already_done:
        n_npy = len(list((outdir / "cutouts").glob("*.npy"))) if (outdir / "cutouts").exists() else 0
        has_fp = (outdir / "manifest_fp.csv").exists()
        print(f"  → already on disk ({n_npy} .npy files, fp={'yes' if has_fp else 'no'}) — skipping.")
        status = "skipped"
    else:
        try:
            download_full_cutouts(
                dia_object_id=obj_id,
                outdir=outdir,
                skip_existing=True,
            )
            status = "downloaded"
        except Exception as exc:
            print(f"  ERROR: {exc}")
            status = f"error: {exc}"

    results.append(
        {
            "diaObjectId": obj_id,
            "gaia_origin": origin,
            "gaia_G_mag": g_mag,
            "n_alerts": n_alerts,
            "outdir": str(outdir),
            "status": status,
        }
    )

print(f"\n{'=' * 60}")
print("All downloads complete.")

## 4. Summary

In [ ]:
from IPython.display import display
import glob as _glob

df_results = pd.DataFrame(results)
display(df_results)

print("\nStatus counts:")
print(df_results["status"].value_counts().to_string())

print("\nDisk footprint:")
for _, row in df_results.iterrows():
    npy_files = _glob.glob(os.path.join(row["outdir"], "cutouts", "*.npy"))
    total_mb = sum(os.path.getsize(f) for f in npy_files) / 1e6 if npy_files else 0
    has_fp = os.path.exists(os.path.join(row["outdir"], "manifest_fp.csv"))
    print(
        f"  {row['diaObjectId']}  {len(npy_files):4d} .npy  {total_mb:6.1f} MB  fp={'✓' if has_fp else '✗'}"
    )